In [1]:
import pandas as pd
from sentence_transformers import SentenceTransformer
import chromadb
import json

# Initialize the Local Setup
# This creates a folder named 'pels_vector_db' in your current directory.
chroma_client = chromadb.PersistentClient(path="./pels_vector_db")

# Create a collection (think of it like an SQL table)
collection = chroma_client.get_or_create_collection(name="prompt_examples")

# Load a fast, lightweight local transformer model. 
# It will download once (~90MB) and then run entirely locally.
print("Loading local embedding model...")
model = SentenceTransformer('all-MiniLM-L6-v2') 
print("Model loaded!")

Loading local embedding model...


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

c:\Users\DELL\AppData\Local\Programs\Python\Python312\Lib\site-packages\huggingface_hub\file_download.py:138: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\DELL\.cache\huggingface\hub\models--sentence-transformers--all-MiniLM-L6-v2. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

...

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Model loaded!


In [5]:
print("Loading Excel file...")
# Using your specific file path
df = pd.read_excel(r"D:/PROJECTS/PELS2/Database/Trinetri Datsets/PELS_CAREER_Extended_v2.xlsx")

# Clean up empty rows/NaNs
df = df.dropna(how='all')

documents = []
metadatas = []
ids = []

print("Formatting rows for embedding...")
for index, row in df.iterrows():
    # Drop empty columns for this specific row to keep data clean
    row_dict = row.dropna().to_dict()
    
    # THE TRICK: Convert the row dictionary into a JSON string.
    content_string = json.dumps(row_dict)
    documents.append(content_string)
    
    # Metadata lets you hard-filter later (e.g., "Only search within the 'CODE' domain")
    metadata = {
        "domain": str(row_dict.get("domain", "UNKNOWN")),
        "skill_level": str(row_dict.get("skill_level", "UNKNOWN"))
    }
    metadatas.append(metadata)
    
    # Give each row a unique ID
    ids.append(f"prompt_{index}")

print(f"Successfully processed {len(documents)} rows from Excel.")

Loading Excel file...
Formatting rows for embedding...
Successfully processed 476 rows from Excel.


In [6]:
print(f"Generating embeddings for {len(documents)} rows (This might take a minute)...")
embeddings = model.encode(documents).tolist()

print("Saving to local Vector DB...")
collection.add(
    embeddings=embeddings,
    documents=documents,
    metadatas=metadatas,
    ids=ids
)

print("Success! Your local vector database is ready.")

Generating embeddings for 476 rows (This might take a minute)...
Saving to local Vector DB...
Success! Your local vector database is ready.


In [8]:
import json

# Define a fake student prompt to test the search
test_student_prompt = "Act as an HR manager. Review my resume for a Data Scientist role and give me feedback on what to improve."

print(f"Searching for prompts similar to: '{test_student_prompt}'\n")

# 1. Embed the test prompt
query_embedding = model.encode([test_student_prompt]).tolist()

# 2. Query the database
results = collection.query(
    query_embeddings=query_embedding,
    n_results=2,
    where={"domain": "CAREER"} 
)

# 3. Display the results clearly
print("--- TOP MATCHES FOUND ---\n")
for idx, document_string in enumerate(results['documents'][0]):
    doc_data = json.loads(document_string)
    
    print(f"Match {idx + 1}:")
    print(f"Prompt ID: {doc_data.get('prompt_id', 'N/A')}")
    print(f"Skill Level: {doc_data.get('skill_level', 'N/A')}")
    print(f"Final Score: {doc_data.get('final_score', 'N/A')}")
    
    # Grab the individual column scores
    c1 = doc_data.get('c1_score', 'N/A')
    c2 = doc_data.get('c2_score', 'N/A')
    c3 = doc_data.get('c3_score', 'N/A')
    c4 = doc_data.get('c4_score', 'N/A')
    c5 = doc_data.get('c5_score', 'N/A')
    c6 = doc_data.get('c6_score', 'N/A')
    
    print(f"Scores Breakdown: C1:{c1} | C2:{c2} | C3:{c3} | C4:{c4} | C5:{c5} | C6:{c6}")
    
    # Print the actual text and justification
    print(f"\nPrompt Text:\n{doc_data.get('prompt_text', 'N/A')}")
    print(f"\nJustification:\n{doc_data.get('justification', 'N/A')}")
    print("-" * 60 + "\n")

Searching for prompts similar to: 'Act as an HR manager. Review my resume for a Data Scientist role and give me feedback on what to improve.'

--- TOP MATCHES FOUND ---

Match 1:
Prompt ID: PELS-L1-CAREER-269
Skill Level: strong
Final Score: 84.0
Scores Breakdown: C1:9 | C2:10 | C3:7 | C4:7 | C5:9 | C6:9

Prompt Text:
You are a career counselor. Give me advanced tips for a successful job search in the competitive tech industry. Include strategies for networking, tailoring applications, leveraging referrals, and preparing for technical and behavioral interviews. Also, provide tips on how to follow up after interviews and negotiate job offers.

Justification:
Multiple techniques composed deliberately, fully adapted to domain. Prompt: 'You are a career counselor. Give me advanced tips for a successful job search in the competitive tech industry. Include strategies for networking, tailoring applications, leveraging referrals, and preparing for technical and behavioral interviews. Also, pro

In [10]:
import pandas as pd
from sentence_transformers import SentenceTransformer
import chromadb
import json
import os
import uuid

# 1. Connect to your EXISTING local database
print("Connecting to existing database...")
chroma_client = chromadb.PersistentClient(path="./pels_vector_db")
collection = chroma_client.get_collection(name="prompt_examples")

# 2. Load the model
print("Loading model...")
model = SentenceTransformer('all-MiniLM-L6-v2')

# 3. List your new files here
new_files = [
    r'D:/PROJECTS/PELS2/Database/Trinetri Datsets/PELS_L1_CODE_Dataset_Extended (1).xlsx',
    r'D:/PROJECTS/PELS2/Database/Trinetri Datsets/PELS_L1_CODE_DSA_Dataset_final.xlsx',
    r'D:/PROJECTS/PELS2/Database/Trinetri Datsets/PELS_L1_CREATE_Dataset_Final.xlsx',
    r'D:/PROJECTS/PELS2/Database/Trinetri Datsets/PELS_L1_GEN_Dataset_Updated_Extended.xlsx'
]

# 4. Process each file
for file_path in new_files:
    file_name = os.path.basename(file_path)
    file_prefix = os.path.splitext(file_name)[0]

    print(f"\n--- Processing {file_name} ---")
    
    # Read the excel file
    try:
        df = pd.read_excel(file_path)
        df = df.dropna(how='all')
    except Exception as e:
        print(f"Error reading {file_path}: {e}")
        continue

    documents = []
    metadatas = []
    ids = []
    
    for index, row in df.iterrows():
        row_dict = row.dropna().to_dict()
        
        # Convert row to JSON string
        content_string = json.dumps(row_dict)
        documents.append(content_string)
        
        # Metadata
        metadata = {
            "domain": str(row_dict.get("domain", "UNKNOWN")),
            "skill_level": str(row_dict.get("skill_level", "UNKNOWN"))
        }
        metadatas.append(metadata)
        
        # UNIQUE ID (prevents duplicate errors on reruns)
        unique_id = f"{file_prefix}_row_{index}_{uuid.uuid4().hex[:8]}"
        ids.append(unique_id)

    if not documents:
        print("No valid rows found in this file. Skipping.")
        continue

    # Generate embeddings
    print(f"Generating embeddings for {len(documents)} rows...")
    embeddings = model.encode(documents).tolist()

    # Add to DB
    print("Adding to database...")
    collection.add(
        embeddings=embeddings,
        documents=documents,
        metadatas=metadatas,
        ids=ids
    )

    print(f"Successfully added {len(documents)} rows from {file_name}!")

# Final count
total_items = collection.count()
print(f"\n✅ All files processed! Your Vector DB now contains {total_items} prompts.")

Connecting to existing database...
Loading model...


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.



--- Processing PELS_L1_CODE_Dataset_Extended (1).xlsx ---
Generating embeddings for 169 rows...
Adding to database...
Successfully added 169 rows from PELS_L1_CODE_Dataset_Extended (1).xlsx!

--- Processing PELS_L1_CODE_DSA_Dataset_final.xlsx ---
Generating embeddings for 50 rows...
Adding to database...
Successfully added 50 rows from PELS_L1_CODE_DSA_Dataset_final.xlsx!

--- Processing PELS_L1_CREATE_Dataset_Final.xlsx ---
No valid rows found in this file. Skipping.

--- Processing PELS_L1_GEN_Dataset_Updated_Extended.xlsx ---
Generating embeddings for 360 rows...
Adding to database...
Successfully added 360 rows from PELS_L1_GEN_Dataset_Updated_Extended.xlsx!

✅ All files processed! Your Vector DB now contains 1055 prompts.
